# MoveBreak Iteration 3 — Footfall Model Tuning

This notebook tunes the existing Random Forest footfall model **without downloading the open data again**. It expects the output folder produced by the earlier modelling notebook.

Required files:

- `movebreak_footfall_model_outputs/model_ready_data.csv`
- `movebreak_footfall_model_outputs/footfall_model.joblib`

The saved model was refitted on all model-ready observations for deployment, including the historical test period. Therefore, it must **not** be evaluated directly on that same test set. This notebook reads its feature contract, then rebuilds an untuned baseline and tuned candidate using leakage-safe chronological splits.

### Evaluation design

1. **Sub-training:** earliest observations up to 56 days before the end.
2. **Validation:** the next 28 days, used to select hyperparameters.
3. **Final test:** the last 28 days, untouched until parameter selection is complete.

The tuning objective is validation MAE. The tuned candidate is accepted before opening the test set only if it improves validation MAE by at least 0.5%.

In [ ]:
# Uncomment and run once if the required packages are missing.
# %pip install pandas numpy matplotlib scikit-learn joblib

import json
import platform
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_SEED = 42
N_PARAMETER_TRIALS = 24
VALIDATION_DAYS = 28
TEST_DAYS = 28
MIN_VALIDATION_IMPROVEMENT_PCT = 0.5

OUTPUT_DIR = Path("movebreak_footfall_model_outputs")
DATA_PATH = OUTPUT_DIR / "model_ready_data.csv"
SAVED_MODEL_PATH = OUTPUT_DIR / "footfall_model.joblib"
TUNING_DIR = OUTPUT_DIR / "tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Missing {DATA_PATH}. Run the footfall model notebook first."
assert SAVED_MODEL_PATH.exists(), f"Missing {SAVED_MODEL_PATH}. Run the footfall model notebook first."

print("Versions:", {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
})

## 1. Load the saved feature contract and model-ready data

`footfall_model.joblib` is loaded only to recover the documented feature schema and original model metadata. The saved estimator itself is not used for test scoring because it was refitted on all available data.

In [ ]:
saved_artifact = joblib.load(SAVED_MODEL_PATH)
assert isinstance(saved_artifact, dict) and "feature_columns" in saved_artifact
feature_columns = list(saved_artifact["feature_columns"])

data = pd.read_csv(DATA_PATH, low_memory=False)
required_columns = set(feature_columns) | {"time_local", "sensor_id", "count", "count_lag_168"}
missing_columns = required_columns - set(data.columns)
assert not missing_columns, f"Missing model-ready columns: {sorted(missing_columns)}"

# Parse mixed daylight-saving offsets safely through UTC, then convert back to Melbourne time.
data["time_local"] = pd.to_datetime(data["time_local"], errors="coerce", utc=True).dt.tz_convert(
    "Australia/Melbourne"
)
data["sensor_id"] = pd.to_numeric(data["sensor_id"], errors="coerce").astype("Int64")
data["sensor_id_cat"] = data["sensor_id_cat"].astype("string")
data["count"] = pd.to_numeric(data["count"], errors="coerce")
data = data.dropna(subset=["time_local", "sensor_id", "count"] + feature_columns).copy()
data = data.sort_values(["time_local", "sensor_id"]).reset_index(drop=True)

assert not data.duplicated(["sensor_id", "time_local"]).any()
assert len(data) >= 1000
assert (data["count"] >= 0).all()

print("Saved model name:", saved_artifact.get("model_name"))
print("Feature columns:", feature_columns)
print("Rows:", len(data), "Sensors:", sorted(data["sensor_id"].astype(int).unique()))
print("Window:", data["time_local"].min(), "to", data["time_local"].max())

## 2. Create chronological sub-training, validation and test periods

The split is based on timestamps rather than row positions, so every sensor uses the same calendar boundaries.

In [ ]:
latest_time = data["time_local"].max()
test_cutoff = latest_time.floor("D") - pd.Timedelta(days=TEST_DAYS - 1)
validation_cutoff = test_cutoff - pd.Timedelta(days=VALIDATION_DAYS)

subtrain = data[data["time_local"] < validation_cutoff].copy()
validation = data[
    (data["time_local"] >= validation_cutoff) & (data["time_local"] < test_cutoff)
].copy()
test = data[data["time_local"] >= test_cutoff].copy()
full_train = data[data["time_local"] < test_cutoff].copy()

assert not subtrain.empty and not validation.empty and not test.empty
assert subtrain["time_local"].max() < validation["time_local"].min()
assert validation["time_local"].max() < test["time_local"].min()
assert set(test["sensor_id"].unique()) <= set(subtrain["sensor_id"].unique())

split_audit = pd.DataFrame([
    {"split": "Sub-training", "start": subtrain.time_local.min(), "end": subtrain.time_local.max(), "rows": len(subtrain)},
    {"split": "Validation", "start": validation.time_local.min(), "end": validation.time_local.max(), "rows": len(validation)},
    {"split": "Final test", "start": test.time_local.min(), "end": test.time_local.max(), "rows": len(test)},
])
display(split_audit)
print("Validation cutoff:", validation_cutoff)
print("Final test cutoff:", test_cutoff)
split_audit.to_csv(TUNING_DIR / "split_audit.csv", index=False)

## 3. Define the baseline pipeline and tuning space

The preprocessing contract matches the original model. Sensor ID is one-hot encoded; numeric variables use median imputation. Random search covers tree count, depth, leaf size, feature subsampling and row subsampling.

In [ ]:
categorical_features = ["sensor_id_cat"]
numeric_features = [column for column in feature_columns if column not in categorical_features]

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_forest_pipeline(parameters):
    preprocessor = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("sensor", make_one_hot_encoder(), categorical_features),
    ])
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            random_state=RANDOM_SEED,
            n_jobs=-1,
            **parameters,
        )),
    ])

BASELINE_PARAMETERS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 3,
    "max_features": 0.8,
    "max_samples": None,
}

PARAMETER_DISTRIBUTIONS = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [None, 12, 20, 30],
    "min_samples_leaf": [1, 2, 3, 5, 10],
    "max_features": [0.5, 0.8, 1.0],
    "max_samples": [None, 0.8],
}

parameter_trials = list(ParameterSampler(
    PARAMETER_DISTRIBUTIONS,
    n_iter=N_PARAMETER_TRIALS,
    random_state=RANDOM_SEED,
))
assert BASELINE_PARAMETERS not in parameter_trials or len(parameter_trials) > 1
print("Random parameter trials:", len(parameter_trials))

## 4. Evaluate the fixed baseline on validation

This creates the reference that tuning must beat. The final test remains unopened.

In [ ]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.abs(actual).sum()
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "WAPE_pct": 100 * np.abs(actual - predicted).sum() / denominator if denominator else np.nan,
        "R2": r2_score(actual, predicted),
    }

X_subtrain, y_subtrain = subtrain[feature_columns], subtrain["count"]
X_validation, y_validation = validation[feature_columns], validation["count"]

baseline_validation_model = make_forest_pipeline(BASELINE_PARAMETERS)
started = time.time()
baseline_validation_model.fit(X_subtrain, y_subtrain)
baseline_validation_prediction = np.clip(
    baseline_validation_model.predict(X_validation), 0, None
)
baseline_validation_metrics = regression_metrics(y_validation, baseline_validation_prediction)
print("Baseline validation metrics:", baseline_validation_metrics)
print(f"Baseline fit time: {time.time() - started:.1f}s")

## 5. Random hyperparameter search on validation

Each candidate is fitted only on sub-training observations and ranked by validation MAE. Progress is printed so a long run remains observable.

In [ ]:
trial_rows = []
for trial_number, parameters in enumerate(parameter_trials, start=1):
    candidate = make_forest_pipeline(parameters)
    started = time.time()
    candidate.fit(X_subtrain, y_subtrain)
    prediction = np.clip(candidate.predict(X_validation), 0, None)
    row = {
        "trial": trial_number,
        **parameters,
        **regression_metrics(y_validation, prediction),
        "fit_seconds": time.time() - started,
    }
    trial_rows.append(row)
    print(
        f"Trial {trial_number:02d}/{len(parameter_trials)} "
        f"MAE={row['MAE']:.3f} time={row['fit_seconds']:.1f}s"
    )

tuning_results = pd.DataFrame(trial_rows).sort_values("MAE").reset_index(drop=True)
best_parameters = {}
integer_parameter_names = {"n_estimators", "max_depth", "min_samples_leaf"}
for key in PARAMETER_DISTRIBUTIONS:
    value = tuning_results.iloc[0][key]
    # A DataFrame may coerce optional integer parameters such as max_depth
    # into floating-point values. Restore the exact sklearn parameter types.
    if pd.isna(value):
        best_parameters[key] = None
    elif key in integer_parameter_names:
        best_parameters[key] = int(value)
    else:
        best_parameters[key] = float(value)

best_validation_mae = float(tuning_results.iloc[0]["MAE"])
baseline_validation_mae = float(baseline_validation_metrics["MAE"])
validation_improvement_pct = 100 * (
    baseline_validation_mae - best_validation_mae
) / baseline_validation_mae

display(tuning_results.head(10))
print("Best parameters:", best_parameters)
print(f"Validation MAE improvement: {validation_improvement_pct:+.2f}%")
tuning_results.to_csv(TUNING_DIR / "parameter_trials.csv", index=False)

## 6. Pre-commit the selected configuration

Selection occurs before final-test evaluation. If tuning improves validation MAE by less than 0.5%, the original baseline configuration is retained to avoid adopting noise.

In [ ]:
use_tuned_configuration = validation_improvement_pct >= MIN_VALIDATION_IMPROVEMENT_PCT
selected_label = "Tuned Random Forest" if use_tuned_configuration else "Baseline Random Forest"
selected_parameters = best_parameters if use_tuned_configuration else BASELINE_PARAMETERS

selection_audit = {
    "selected_model": selected_label,
    "minimum_required_validation_improvement_pct": MIN_VALIDATION_IMPROVEMENT_PCT,
    "actual_validation_improvement_pct": validation_improvement_pct,
    "selected_parameters": selected_parameters,
}
print(json.dumps(selection_audit, indent=2))
(TUNING_DIR / "selection_before_test.json").write_text(
    json.dumps(selection_audit, indent=2), encoding="utf-8"
)

## 7. Refit on full training data and open the final test once

The chosen configuration is now trained on sub-training plus validation. The final 28-day test is evaluated once and compared with the fixed baseline and seasonal-naive benchmark for reporting.

In [ ]:
X_full_train, y_full_train = full_train[feature_columns], full_train["count"]
X_test, y_test = test[feature_columns], test["count"]

selected_test_model = make_forest_pipeline(selected_parameters)
selected_test_model.fit(X_full_train, y_full_train)
selected_test_prediction = np.clip(selected_test_model.predict(X_test), 0, None)

baseline_test_model = make_forest_pipeline(BASELINE_PARAMETERS)
baseline_test_model.fit(X_full_train, y_full_train)
baseline_test_prediction = np.clip(baseline_test_model.predict(X_test), 0, None)
seasonal_prediction = test["count_lag_168"].to_numpy(dtype=float)

test_metric_rows = [
    {"model": selected_label, **regression_metrics(y_test, selected_test_prediction)},
]
if use_tuned_configuration:
    test_metric_rows.append({
        "model": "Fixed baseline Random Forest",
        **regression_metrics(y_test, baseline_test_prediction),
    })
test_metric_rows.append({
    "model": "Seasonal naive",
    **regression_metrics(y_test, seasonal_prediction),
})
test_metrics = pd.DataFrame(test_metric_rows).sort_values("MAE").reset_index(drop=True)
display(test_metrics.round({"MAE": 2, "RMSE": 2, "WAPE_pct": 2, "R2": 3}))

test_predictions = test[["time_local", "sensor_id", "count"]].copy()
test_predictions["selected_prediction"] = selected_test_prediction
test_predictions["fixed_baseline_prediction"] = baseline_test_prediction
test_predictions["seasonal_prediction"] = seasonal_prediction
test_predictions["selected_absolute_error"] = np.abs(
    test_predictions["count"] - test_predictions["selected_prediction"]
)

sensor_metrics = []
for sensor_id, group in test_predictions.groupby("sensor_id"):
    sensor_comparisons = [(selected_label, "selected_prediction")]
    if use_tuned_configuration:
        sensor_comparisons.append(("Fixed baseline Random Forest", "fixed_baseline_prediction"))
    sensor_comparisons.append(("Seasonal naive", "seasonal_prediction"))
    for label, column in sensor_comparisons:
        sensor_metrics.append({
            "sensor_id": int(sensor_id),
            "model": label,
            **regression_metrics(group["count"], group[column]),
        })
sensor_metrics = pd.DataFrame(sensor_metrics)
display(sensor_metrics.sort_values(["sensor_id", "MAE"]).round(3))

test_metrics.to_csv(TUNING_DIR / "final_test_metrics.csv", index=False)
sensor_metrics.to_csv(TUNING_DIR / "final_test_metrics_by_sensor.csv", index=False)
test_predictions.to_csv(TUNING_DIR / "final_test_predictions.csv", index=False)

## 8. Visualise validation search and final-test comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

top_trials = tuning_results.head(10).sort_values("MAE", ascending=True)
axes[0].barh(top_trials["trial"].astype(str), top_trials["MAE"])
axes[0].axvline(baseline_validation_mae, color="red", linestyle="--", label="Fixed baseline")
axes[0].set(title="Top validation trials", xlabel="Validation MAE", ylabel="Trial")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.2)

metric_plot = test_metrics.sort_values("MAE", ascending=False)
axes[1].barh(metric_plot["model"], metric_plot["MAE"])
axes[1].set(title="Final-test model comparison", xlabel="Test MAE")
axes[1].grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

plot_start = test_predictions["time_local"].max() - pd.Timedelta(days=7)
sensor_ids = sorted(test_predictions["sensor_id"].astype(int).unique())
fig, axes = plt.subplots(len(sensor_ids), 1, figsize=(15, 3.5 * len(sensor_ids)), sharex=True)
axes = np.atleast_1d(axes)
for axis, sensor_id in zip(axes, sensor_ids):
    plot_data = test_predictions[
        (test_predictions["sensor_id"] == sensor_id)
        & (test_predictions["time_local"] >= plot_start)
    ]
    axis.plot(plot_data["time_local"], plot_data["count"], label="Actual", linewidth=1.5)
    axis.plot(plot_data["time_local"], plot_data["selected_prediction"], label=selected_label, linewidth=1.2)
    axis.set_title(f"Sensor {sensor_id}: final seven test days")
    axis.set_ylabel("Pedestrians/hour")
    axis.grid(alpha=0.2)
    axis.legend()
axes[-1].set_xlabel("Melbourne local time")
plt.tight_layout()
plt.show()

## 9. Refit the pre-selected configuration on all data and export

Only after evaluation is complete is the selected configuration refitted on all model-ready rows for deployment. The exported artifact preserves the same inference contract as the original model.

In [ ]:
deployment_model = make_forest_pipeline(selected_parameters)
deployment_model.fit(data[feature_columns], data["count"])

deployment_artifact = {
    "model": deployment_model,
    "feature_columns": feature_columns,
    "model_name": selected_label,
    "parameters": selected_parameters,
    "sensor_ids": sorted(data["sensor_id"].astype(int).unique().tolist()),
    "timezone": "Australia/Melbourne",
    "target": "hourly pedestrian count",
    "selection_rule": selection_audit,
}
tuned_artifact_path = TUNING_DIR / "footfall_model_selected.joblib"
joblib.dump(deployment_artifact, tuned_artifact_path)

# Verify serialization with five real feature rows.
smoke_features = data.tail(5)[feature_columns]
before_save = np.clip(deployment_model.predict(smoke_features), 0, None)
loaded_artifact = joblib.load(tuned_artifact_path)
after_load = np.clip(loaded_artifact["model"].predict(smoke_features), 0, None)
assert np.allclose(before_save, after_load)

tuning_metadata = {
    "selected_model": selected_label,
    "selected_parameters": selected_parameters,
    "validation_improvement_pct": validation_improvement_pct,
    "validation_cutoff": str(validation_cutoff),
    "test_cutoff": str(test_cutoff),
    "feature_columns": feature_columns,
    "random_seed": RANDOM_SEED,
    "parameter_trials": N_PARAMETER_TRIALS,
    "versions": {"pandas": pd.__version__, "scikit_learn": sklearn.__version__},
}
(TUNING_DIR / "tuning_metadata.json").write_text(
    json.dumps(tuning_metadata, indent=2), encoding="utf-8"
)

print("Selected model:", selected_label)
print("Selected parameters:", selected_parameters)
print("Exported:", tuned_artifact_path)
print("Model artifact round-trip test passed.")

## Reporting guidance

- Report the validation improvement used to select or reject the tuned configuration.
- Report final-test MAE, RMSE, WAPE and R² beside the seasonal-naive benchmark.
- Do not describe R² as accuracy.
- If validation improvement is below 0.5%, retaining the fixed baseline is a valid result.
- If final-test improvement is small, conclude that additional event, holiday or rainfall features may offer more value than further tree tuning.
- This remains a rolling one-hour-ahead model because it relies heavily on recent observed counts.